In [4]:
import pandas as pd
from database.connection import get_connection
import psycopg2
import os

In [1]:
from etl.extract import (
    load_known_platforms,
    get_video_ids_since_date,
    fetch_video_metadata
)
from config.settings import Settings

In [2]:
known_platforms = load_known_platforms()

all_video_ids = []

for channel_id in Settings.CHANNEL_IDS:
    ids = get_video_ids_since_date(channel_id)
    all_video_ids.extend(ids)

rows = fetch_video_metadata(
    all_video_ids,
    known_platforms
)

Loaded 412 cached platforms from DB
Detected aHg5BZamsSs -> Regular
Detected ijAoGe0fo8I -> Regular
Detected vwBVBrVTHbM -> Short
Detected XX191Zl2VWc -> Regular
Detected p2ZoTwk2CN0 -> Regular
Detected 7gjvdB3eITA -> Regular
Detected ixFhMqkdeXA -> Regular
Detected 0Oca32nLOMQ -> Regular
Detected uKqQQkJwk4E -> Regular
Detected WNXHB1aunEU -> Regular
Fetched 50 videos (quota: 4)
Fetched 50 videos (quota: 5)
Detected -jP80rIp8m8 -> Regular
Detected ryiBkOF2-qQ -> Short
Detected euYqGAdfPZM -> Regular
Detected 80J_YOQFOJ8 -> Regular
Detected 3ynTkkmsSFc -> Short
Fetched 50 videos (quota: 6)
Fetched 50 videos (quota: 7)
Detected SyItlk8r1fk -> Regular
Detected lQgMKU1JZ-Q -> Regular
Detected oWPn_W4bpu8 -> Regular
Detected VX8BRSOeKw4 -> Regular
Detected 38qPgg_MDYI -> Regular
Detected u7J0L4pmPvs -> Regular
Detected 254dMdynpak -> Regular
Detected QexWNrl7su0 -> Regular
Detected o5QFb-WujEU -> Regular
Detected PUDsvxQhENE -> Regular
Detected RabVSZD7LVM -> Regular
Detected j3e1Bvt7WzM -

In [5]:
df_raw = pd.DataFrame(rows)
display(df_raw)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,aHg5BZamsSs,MMC probing fake MC syndicate amid alleged ins...,The Malaysian Medical Council (MMC) will lead ...,"['news', 'the star tv', 'the star', 'thestartv...",25,The Star,2026-06-20T14:14:41Z,240,6,0,Regular,2026-06-21 00:39:09.499079,PT5M55S
1,ijAoGe0fo8I,One killed and several injured in train collis...,A train driver was killed and dozens more inju...,"['news', 'the star tv', 'the star', 'thestartv...",25,The Star,2026-06-20T14:03:59Z,506,5,3,Regular,2026-06-21 00:39:09.499079,PT1M46S
2,vwBVBrVTHbM,Here are the top 3 #StatePolls2026 stories for...,Stay up to date with our daily 2026 state poll...,"['news', 'the star tv', 'the star']",25,The Star,2026-06-20T13:16:16Z,628,2,0,Short,2026-06-21 00:39:09.499079,PT29S
3,XX191Zl2VWc,PM Anwar backs Nallini to strengthen Media Cou...,Prime Minister Datuk Seri Anwar Ibrahim expres...,"['news', 'the star tv', 'the star', 'thestartv...",25,The Star,2026-06-20T10:39:08Z,515,10,5,Regular,2026-06-21 00:39:09.499079,PT59S
4,p2ZoTwk2CN0,RM1mil allocation for media workers under Tabu...,The government has allocated RM1mil for Tabung...,"['news', 'the star tv', 'the star', 'thestartv...",25,The Star,2026-06-20T10:16:08Z,199,1,0,Regular,2026-06-21 00:39:09.499079,PT3M6S
...,...,...,...,...,...,...,...,...,...,...,...,...,...
444,wXu4eP-H3Z0,4 Sekeluarga Maut: Proses Bedah Siasat Dijalan...,"Buletin Pagi siaran Isnin, 15 Jun 2026 bersama...",[],25,Buletin TV3,2026-06-15T02:02:13Z,24709,172,5,Regular,2026-06-21 00:39:09.499079,PT28M38S
445,xj3GMetPBjo,NAHAS UDARA | 12 Maut Pesawat Penerjun Udara T...,Sekurang-kurangnya 12 maut apabila sebuah pesa...,"['BU', 'Buletin TV3', 'buletin utama', 'buleti...",25,Buletin TV3,2026-06-15T01:55:44Z,9762,77,2,Regular,2026-06-21 00:39:09.499079,PT47S
446,W3fR4UiEc4Y,"ISU ROHINGYA | Ada Mohon Jadi AJK Kampung, Lak...",Kewujudan komuniti Rohingya di beberapa kawasa...,"['BU', 'Buletin TV3', 'buletin utama', 'buleti...",25,Buletin TV3,2026-06-15T01:52:40Z,6276,77,103,Regular,2026-06-21 00:39:09.499079,PT1M26S
447,632XYD6l4n0,PERANG ASIA BARAT | AS-Iran Capai Perjanjian D...,Amerika Syarikat dan Iran mencapai persetujuan...,"['BU', 'Buletin TV3', 'buletin utama', 'buleti...",25,Buletin TV3,2026-06-15T01:51:10Z,1701,22,12,Regular,2026-06-21 00:39:09.499079,PT1M35S


In [3]:
from psycopg2.extras import execute_values
from database.connection import get_connection

In [7]:
conn = get_connection()
cur = conn.cursor()

In [11]:
from psycopg2.extras import execute_values


In [12]:
query = """
    INSERT INTO public.video_metrics_time_series (
        video_id,
        title,
        description,
        tags,
        category_id,
        channel_title,
        published_at,
        views,
        likes,
        comments_count,
        platform,
        fetched_at,
        duration
    ) VALUES %s
"""

execute_values(cur, query, rows, page_size=5000)

In [35]:
conn.rollback()

In [36]:
cur.execute(
    """
    ALTER TABLE public.video_metrics_time_series
        DROP COLUMN IF EXISTS fetched_at_sgt,
        DROP COLUMN IF EXISTS published_at_sgt
    """
)


In [13]:
df = pd.read_sql(
    "SELECT * FROM video_metrics_time_series",
    conn
)

C:\Users\User\AppData\Local\Temp\ipykernel_27060\4205903146.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [14]:
display(df)

,id,video_id,title,description,tags,category_id,channel_title,published_at,views,likes,comments_count,platform,fetched_at,duration
0,2576,aHg5BZamsSs,MMC probing fake MC syndicate amid alleged ins...,The Malaysian Medical Council (MMC) will lead ...,"['news', 'the star tv', 'the star', 'thestartv...",25,The Star,2026-06-20 14:14:41,240,6,0,Regular,2026-06-21 00:39:09,PT5M55S
1,2577,ijAoGe0fo8I,One killed and several injured in train collis...,A train driver was killed and dozens more inju...,"['news', 'the star tv', 'the star', 'thestartv...",25,The Star,2026-06-20 14:03:59,506,5,3,Regular,2026-06-21 00:39:09,PT1M46S
2,2578,vwBVBrVTHbM,Here are the top 3 #StatePolls2026 stories for...,Stay up to date with our daily 2026 state poll...,"['news', 'the star tv', 'the star']",25,The Star,2026-06-20 13:16:16,628,2,0,Short,2026-06-21 00:39:09,PT29S
3,2579,XX191Zl2VWc,PM Anwar backs Nallini to strengthen Media Cou...,Prime Minister Datuk Seri Anwar Ibrahim expres...,"['news', 'the star tv', 'the star', 'thestartv...",25,The Star,2026-06-20 10:39:08,515,10,5,Regular,2026-06-21 00:39:09,PT59S
4,2580,p2ZoTwk2CN0,RM1mil allocation for media workers under Tabu...,The government has allocated RM1mil for Tabung...,"['news', 'the star tv', 'the star', 'thestartv...",25,The Star,2026-06-20 10:16:08,199,1,0,Regular,2026-06-21 00:39:09,PT3M6S
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1672,1224,v76wvD1vvhQ,Top STPM students to get public university sch...,The country’s top Sijil Tinggi Persekolahan Ma...,"['news', 'the star tv', 'the star', 'thestartv...",25,The Star,2026-06-18 10:26:21,638,8,3,Regular,2026-06-20 04:21:14,PT4M35S
1673,1225,uqA35LurToM,PERJANJIAN AS-IRAN | Majoriti Saham Asia Melon...,Perkembangan terbaharu konflik Asia Barat memb...,"['BU', 'Buletin TV3', 'buletin utama', 'buleti...",25,Buletin TV3,2026-06-18 12:17:25,966,15,2,Regular,2026-06-20 04:21:14,PT1M34S
1674,1226,OGSa1qJX6Yw,"PERANG ASIA BARAT | Netanyahu Tetap Berdegil, ...","Pemimpin rejim, Benjamin Netanyahu tetap berde...","['BU', 'Buletin TV3', 'buletin utama', 'buleti...",25,Buletin TV3,2026-06-16 02:09:17,5333,38,42,Regular,2026-06-20 04:21:14,PT54S
1675,1227,xXx9Do6zGAM,MAHKAMAH | Wanita Didakwa Bunuh Warga Indonesi...,Seorang suri rumah berdepan tuduhan membunuh s...,"['BU', 'Buletin TV3', 'buletin utama', 'buleti...",25,Buletin TV3,2026-06-19 12:34:19,887,13,1,Regular,2026-06-20 04:21:14,PT51S


In [17]:
cursor.execute(
    """
    UPDATE public.video_metrics_time_series
    SET
        published_at_sgt = published_at AT TIME ZONE 'UTC' AT TIME ZONE 'Asia/Singapore',
        fetched_at_sgt = fetched_at AT TIME ZONE 'UTC' AT TIME ZONE 'Asia/Singapore';
    """
)

In [15]:
cur.close()
conn.close()

In [2]:
import logic
print(logic)
print(logic.__file__)

dir(logic)

import os

print(os.listdir(next(iter(logic.__path__))))

print(logic.__path__)

<module 'logic' from 'C:\\Users\\User\\Desktop\\YT_TimeSeries\\logic\\__init__.py'>
C:\Users\User\Desktop\YT_TimeSeries\logic\__init__.py
['time_series_collection_optimized.py', 'time_series_collection_supabase.py', '__init__.py', '__pycache__']
['C:\\Users\\User\\Desktop\\YT_TimeSeries\\logic']


In [1]:
from etl.extract import load_known_platforms

platforms = load_known_platforms()

print(platforms)

Loaded 412 cached platforms from DB
{'_i_jedqiZ98': 'Regular', '_Jq93RTajzg': 'Short', '_QfdSSgGPDA': 'Short', '_t1Z2pqkpYI': 'Regular', '_TSfJNZsB-g': 'Regular', '-_EkEWVzWQ8': 'Regular', '-8pErDoPRyU': 'Regular', '-ahlkfVPvDM': 'Regular', '-bSJgXzCfh0': 'Regular', '-jfjUAmu1OM': 'Short', '-lJ2aF-5sEQ': 'Regular', '-M-ZcELjWpA': 'Regular', '-n7VVl_20jI': 'Short', '-QVcHkJqyiE': 'Short', '-QyrOsS47YQ': 'Short', '0bjlGR8UXwg': 'Regular', '0gKlsk_32NQ': 'Regular', '0HLIff4xuMA': 'Regular', '0PH2QSfGJ3I': 'Regular', '0SjkNn9LMp4': 'Regular', '0y-0y9iquak': 'Regular', '0Yi5nlzdrW4': 'Regular', '1dTCJLSaTr4': 'Regular', '1JrSN370q-8': 'Regular', '1Kr4LAMm2og': 'Regular', '2BSEuRPU6e0': 'Short', '2d1MowONYmU': 'Regular', '2F9zv8YXrDw': 'Short', '2zS-jCQ6Ap0': 'Short', '3-PNtLSJRGs': 'Regular', '3c2-5fPlDD4': 'Short', '3CRA2mJPY7E': 'Regular', '3cT0FNmiXjo': 'Regular', '3fT26kAI3A0': 'Regular', '3gyNtKI5MZg': 'Regular', '3KTpdJB6LWA': 'Regular', '3PEbq17t1-U': 'Regular', '3QIRrK6xNhk': 'Regul